In [58]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()

In [59]:
raw_health_rdd = spark.sparkContext.textFile('/content/healthcare_dataset_stroke_data.csv')
raw_health_rdd.take(5)

['id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke',
 '9046,Male,67,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1',
 '51676,Female,61,0,0,Yes,Self-employed,Rural,202.21,N/A,never smoked,1',
 '31112,Male,80,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1',
 '60182,Female,49,0,0,Yes,Private,Urban,171.23,34.4,smokes,1']

In [60]:
def skip_header(idx, iter):
  if idx == 0:
    next(iter) # Skips first row of first partition
  yield from iter # Yield remaining rows

In [61]:
cached_health_rdd = raw_health_rdd.mapPartitionsWithIndex(skip_header).cache()
cached_health_rdd.take(5)

['9046,Male,67,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1',
 '51676,Female,61,0,0,Yes,Self-employed,Rural,202.21,N/A,never smoked,1',
 '31112,Male,80,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1',
 '60182,Female,49,0,0,Yes,Private,Urban,171.23,34.4,smokes,1',
 '1665,Female,79,1,0,Yes,Self-employed,Rural,174.12,24,never smoked,1']

# We will reuse this cached RDD in 2 different use cases

## 1. To get stroke prcentage based on work type


In [ ]:
# Create pair RDD of (work_type, stroke_indicator)
def WorkTypeAndStroke(vals):
  record = vals.split(',')
  work_type = record[6]
  stroke_indicator = int(record[11])
  yield (work_type, stroke_indicator)



# Get count of participants with stroke and total participants
def getNumStrokeAndTotalCount(val1, val2):
  # For subsequent aggregations, val1 is already a tuple (stroke_count, total_count) and val2 can be either a tuple or an integer
  if isinstance(val1, tuple):

    # Unpack the tuple from val1 and initialize stroke_count and total_count
    stroke_count, total_count = val1

    # Check the type of val2
    # val2 will be a tuple when merging results from different partitions as part of the shuffle phase
    if isinstance(val2, tuple):
      stroke_count += val2[0]
      total_count += val2[1]

    # val2 is an integer for all the aggregations in a specific partition
    else:
      stroke_count += val2
      total_count += 1

  # Both val1 and val2 are integers for the first aggregation of each key
  else:
    stroke_count = val1 + val2
    total_count = 2

  # Return the aggregated stroke count and total count as a tuple
  return (stroke_count, total_count)


# Calculate percentage
def calaculatePercentage(val):
  if isinstance(val, tuple):
    stroke_count, total_count = val
  else:
    stroke_count, total_count = val, 1
  percent = round((stroke_count * 100 / total_count), 2)
  return (stroke_count, total_count, percent)

In [63]:
work_type_and_stroke_rdd = cached_health_rdd.flatMap(WorkTypeAndStroke)
work_type_and_stroke_rdd.take(5)

[('Private', 1),
 ('Self-employed', 1),
 ('Private', 1),
 ('Private', 1),
 ('Self-employed', 1)]

In [64]:
# Below reduceByKey logic only counts participants with stroke but not the total participants. Hence we need to use custom logic
work_type_and_stroke_rdd.reduceByKey(lambda x, y: (x+y)).collect()

[('Private', 149),
 ('Govt_job', 33),
 ('children', 2),
 ('Self-employed', 65),
 ('Never_worked', 0)]

In [65]:
reduced_work_type_and_stroke_rdd = work_type_and_stroke_rdd.reduceByKey(getNumStrokeAndTotalCount)
reduced_work_type_and_stroke_rdd.take(5)

[('Private', (149, 2925)),
 ('Govt_job', (33, 657)),
 ('children', (2, 687)),
 ('Self-employed', (65, 819)),
 ('Never_worked', (0, 22))]

In [66]:
work_type_stroke_percent = reduced_work_type_and_stroke_rdd.mapValues(calaculatePercentage)
work_type_stroke_percent.collect()

[('Private', (149, 2925, 5.09)),
 ('Govt_job', (33, 657, 5.02)),
 ('children', (2, 687, 0.29)),
 ('Self-employed', (65, 819, 7.94)),
 ('Never_worked', (0, 22, 0.0))]

## 2. To get stroke prcentage based on gender

In [67]:
# Create pair RDD of (gender, stroke_indicator)
def GenderAndStroke(vals):
  record = vals.split(',')
  gender = record[1]
  stroke_indicator = int(record[11])
  yield (gender, stroke_indicator)

In [68]:
gender_and_stroke_rdd = cached_health_rdd.flatMap(GenderAndStroke)
gender_and_stroke_rdd.take(5)

[('Male', 1), ('Female', 1), ('Male', 1), ('Female', 1), ('Female', 1)]

In [69]:
reduced_gender_and_stroke_rdd = gender_and_stroke_rdd.reduceByKey(getNumStrokeAndTotalCount)
reduced_gender_and_stroke_rdd.take(5)

[('Male', (108, 2115)), ('Female', (141, 2994)), ('Other', 0)]

In [ ]:
gender_stroke_percent = reduced_gender_and_stroke_rdd.mapValues(calaculatePercentage)
gender_stroke_percent.collect()

[('Male', (108, 2115, 5.11)),
 ('Female', (141, 2994, 4.71)),
 ('Other', (0, 1, 0.0))]